# Подготовка данных

Сначала посмотрим на датасет, затем уберём признаки с утечкой и сохраним чистую таблицу для EDA и обучения.

Прогноз нужен сразу после создания бронирования. Значит, в модель можно передавать только ту информацию, которая известна в этот момент.

## 1. Загрузка данных

In [1]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent

DATA_PATH = ROOT / "data/raw/hotel_bookings.csv"
CLEAN_DATA_PATH = ROOT / "data/interim/hotel_bookings_clean.parquet"

raw = pd.read_csv(DATA_PATH)
raw.head()

,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,...,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,...,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,...,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,...,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,...,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


## 2. Быстрый обзор

In [2]:
print(f"Строки: {raw.shape[0]:,}")
print(f"Признаки: {raw.shape[1]}")
raw.info()

Строки: 119,390
Признаки: 32
<class 'pandas.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 32 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  str    
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  str    
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119386 non-null  float64
 11  babies                          119390 non-null  int64  
 12

In [3]:
missing = pd.DataFrame({
    "missing": raw.isna().sum(),
    "share": raw.isna().mean(),
})

display(missing.query("missing > 0").sort_values("share", ascending=False).round(3))
display(raw["is_canceled"].value_counts().rename_axis("is_canceled").to_frame("bookings"))
print("Полных совпадений строк:", raw.duplicated().sum())

,missing,share
company,112593,0.943
agent,16340,0.137
country,488,0.004
children,4,0.000


,bookings
is_canceled,
0,75166
1,44224


Полных совпадений строк: 31994


В датасете 119 тысяч бронирований. Отмены составляют около 37%, поэтому одной accuracy для оценки моделей будет недостаточно.

Пропуски есть только в `children`, `country`, `agent` и `company`. Полные совпадения встречаются часто, но пока нет смысла их удалять: без `booking_id` нельзя отличить настоящий дубль от двух разных броней с одинаковыми параметрами.

## 3. Удаление leakage

Пять колонок недоступны в момент прогноза:

- `reservation_status` и `reservation_status_date` напрямую описывают итог бронирования;
- `booking_changes`, `assigned_room_type` и `days_in_waiting_list` могут появиться или измениться уже после создания брони.

In [4]:
leakage_columns = [
    "reservation_status",
    "reservation_status_date",
    "booking_changes",
    "assigned_room_type",
    "days_in_waiting_list",
]

clean = raw.drop(columns=leakage_columns).copy()

## 4. Пропуски и типы

In [5]:
clean["country"] = clean["country"].fillna("Unknown")

for column in ["agent", "company"]:
    clean[column] = clean[column].astype("Int64").astype("string")

clean.isna().sum()[lambda values: values > 0]

children         4
agent        16340
company     112593
dtype: int64

`agent` и `company` по смыслу являются категориями, а не числами. Пропуск в `country` удобно обозначить отдельной категорией `Unknown`.

В `children` всего четыре пропуска. Их можно оставить до preprocessing: медиана заполнит их уже внутри модельного pipeline.

## 5. Подозрительные значения

In [6]:
zero_guests = (
    (clean["adults"] == 0)
    & (clean["children"].fillna(0) == 0)
    & (clean["babies"] == 0)
)

pd.Series({
    "rows_with_zero_guests": zero_guests.sum(),
    "min_adr": clean["adr"].min(),
    "median_adr": clean["adr"].median(),
    "max_adr": clean["adr"].max(),
    "max_lead_time": clean["lead_time"].max(),
})

rows_with_zero_guests     180.000
min_adr                    -6.380
median_adr                 94.575
max_adr                  5400.000
max_lead_time             737.000
dtype: float64

В данных встречаются нулевая стоимость, несколько отрицательных значений ADR и 180 бронирований без гостей. Пока удалять такие строки нет смысла: контекста недостаточно, а редкие крайние значения сами по себе не мешают моделям деревьев.

## 6. Сохранение

In [7]:
CLEAN_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
clean.to_parquet(CLEAN_DATA_PATH, index=False)

clean.shape

(119390, 27)

## Итог

В итоге из таблицы ушёл только явный leakage. Спорные строки и выбросы пока остались: данных недостаточно, чтобы уверенно считать их ошибками.